# Basic Data Analysis
### A One-Day Intensive: 2–3 Hour Lesson + 4 Hours Hands-On
**Powered by Google Colaboratory**

From raw files to business insight: load → clean → transform → summarize → visualize → tell the story. **Core libraries:** `pandas`, `numpy`, `matplotlib`, `seaborn`, `re`, `requests`.

---
**How to use this notebook**
1. Run cells top to bottom (`Shift + Enter` runs a cell).
2. The **Setup** section generates all dummy datasets (CSV, TXT, Parquet, and a mock API JSON) — no uploads needed.
3. Lesson sections (Parts 1 onward) are the **2–3 hour instructor-led portion**.
4. The **Hands-On Lab** section at the end is the **4-hour guided exercise portion** with a capstone.

---
> © **Data Engineering Pilipinas (DEP). All rights reserved.**
> This course material was created by and is the property of Data Engineering Pilipinas. It may not be copied, distributed, modified, or used — in whole or in part — without prior written consent from Data Engineering Pilipinas.

## ⚙️ Setup — Generate the Dummy Datasets
Run the cell below **once** at the start of the session. It creates four practice datasets in a `data/` folder:

| File | Format | Simulates |
|---|---|---|
| `sales_data.csv` | CSV | Retail sales orders (intentionally *messy*: missing values, duplicates, mixed date formats, bad emails) |
| `server_logs.txt` | TXT | Raw web-server logs (for **regex** practice) |
| `transactions.parquet` | Parquet | Bank transactions (columnar format) |
| `customers_api.json` | JSON | A saved **API response** with customer records |

In [ ]:
# =============================================================
# SETUP — Run this cell first! It generates all dummy datasets.
# Creates: sales_data.csv, server_logs.txt, transactions.parquet,
#          customers_api.json  (a saved mock API response)
# =============================================================
import pandas as pd
import numpy as np
import json, random, os

np.random.seed(42)
random.seed(42)
os.makedirs("data", exist_ok=True)

# ---------- 1. CSV: retail sales data (intentionally messy) ----------
n = 500
regions  = ["NCR", "Cebu", "Davao", "Iloilo", "Baguio"]
products = ["Rice 25kg", "Cooking Oil 1L", "Instant Coffee",
            "Laundry Soap", "Canned Sardines", "Sugar 1kg"]
dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")

df = pd.DataFrame({
    "order_id":   [f"ORD-{i:05d}" for i in range(1, n + 1)],
    "order_date": np.random.choice(dates, n),
    "region":     np.random.choice(regions, n),
    "product":    np.random.choice(products, n),
    "quantity":   np.random.randint(1, 20, n),
    "unit_price": np.round(np.random.uniform(20, 1500, n), 2),
    "customer_email": [f"customer{i}@{random.choice(['gmail.com','yahoo.com','outlook.ph'])}" for i in range(n)],
    "phone":      [f"09{random.randint(100000000, 999999999)}" for _ in range(n)],
})
df["total_amount"] = (df["quantity"] * df["unit_price"]).round(2)

# Inject "messiness" so we can practice cleaning
dirty = df.copy()
dirty.loc[dirty.sample(25, random_state=1).index, "unit_price"] = np.nan
dirty.loc[dirty.sample(15, random_state=2).index, "region"] = " ncr "
dirty.loc[dirty.sample(10, random_state=3).index, "customer_email"] = "invalid-email"
dirty = pd.concat([dirty, dirty.sample(12, random_state=4)])   # duplicates
dirty["order_date"] = dirty["order_date"].astype(str)
mixed_idx = dirty.sample(20, random_state=5).index
dirty.loc[mixed_idx, "order_date"] = dirty.loc[mixed_idx, "order_date"].str.replace("-", "/")
dirty.to_csv("data/sales_data.csv", index=False)

# ---------- 2. TXT: server log file (for regex practice) ----------
levels    = ["INFO", "WARNING", "ERROR", "DEBUG"]
endpoints = ["/api/orders", "/api/customers", "/api/products", "/login", "/checkout"]
lines = []
for i in range(300):
    ts  = pd.Timestamp("2025-06-01") + pd.Timedelta(seconds=int(np.random.uniform(0, 86400 * 30)))
    lvl = random.choices(levels, weights=[60, 15, 10, 15])[0]
    ip  = f"{random.randint(1,255)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"
    ep  = random.choice(endpoints)
    ms  = random.randint(5, 2500)
    status = random.choices([200, 201, 404, 500, 403], weights=[70, 10, 8, 7, 5])[0]
    lines.append(f"{ts.strftime('%Y-%m-%d %H:%M:%S')} [{lvl}] client={ip} "
                 f"endpoint={ep} status={status} response_time={ms}ms")
with open("data/server_logs.txt", "w") as f:
    f.write("\n".join(sorted(lines)))

# ---------- 3. Parquet: bank-style transactions ----------
m = 800
tx = pd.DataFrame({
    "transaction_id":   [f"TXN{i:07d}" for i in range(1, m + 1)],
    "account_id":       [f"ACC{random.randint(1000,1099)}" for _ in range(m)],
    "transaction_type": np.random.choice(["deposit","withdrawal","transfer","payment"], m, p=[.3,.3,.2,.2]),
    "amount":           np.round(np.random.lognormal(mean=7, sigma=1.2, size=m), 2),
    "channel":          np.random.choice(["mobile_app","atm","branch","online"], m),
    "timestamp":        pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.uniform(0,365,m), unit="D"),
    "is_flagged":       np.random.choice([True, False], m, p=[.04,.96]),
})
tx.to_parquet("data/transactions.parquet", index=False)

# ---------- 4. Mock API response saved as JSON ----------
first = ["Maria","Jose","Juan","Ana","Carlo","Liza","Ramon","Grace","Paolo","Nina"]
last  = ["Santos","Reyes","Cruz","Garcia","Torres","Flores","Ramos","Mendoza"]
customers = []
for i in range(1, 61):
    fn, ln = random.choice(first), random.choice(last)
    customers.append({
        "customer_id": i,
        "name": f"{fn} {ln}",
        "email": f"{fn.lower()}.{ln.lower()}{i}@example.com",
        "city": random.choice(["Quezon City","Makati","Cebu City","Davao City","Taguig"]),
        "signup_date": str((pd.Timestamp("2024-01-01") + pd.Timedelta(days=random.randint(0,700))).date()),
        "loyalty_points": random.randint(0, 5000),
        "is_active": random.random() > 0.15,
    })
with open("data/customers_api.json", "w") as f:
    json.dump({"status": "success", "count": len(customers), "data": customers}, f, indent=2)

print("✅ Setup complete! Files created in ./data/:")
for fn in sorted(os.listdir("data")):
    print("   -", fn)

## Part 1 — Python Essentials (≈30 min)
Python is the *lingua franca* of data work. We only need a focused subset: **variables, collections, control flow, functions**, and the habit of writing small, reusable pieces of logic.

In [ ]:
# Variables & data types
store_name = "DEP Sari-Sari Store"   # str
daily_sales = 15750.50               # float
num_customers = 42                   # int
is_open = True                       # bool

print(type(store_name), type(daily_sales), type(num_customers), type(is_open))
print(f"{store_name} earned ₱{daily_sales:,.2f} from {num_customers} customers today.")

In [ ]:
# Collections: list, tuple, dict, set
products = ["rice", "oil", "coffee", "soap"]            # list  — ordered, mutable
location = (14.6760, 121.0437)                          # tuple — ordered, immutable (e.g., lat/lon)
prices   = {"rice": 52.0, "oil": 85.5, "coffee": 7.5}   # dict  — key/value lookup
regions  = {"NCR", "Cebu", "NCR", "Davao"}              # set   — unique values only

print("3rd product:", products[2])
print("Price of oil:", prices["oil"])
print("Unique regions:", regions)

In [ ]:
# Control flow: if / for / while
sales = [1200, 3400, 800, 5600, 2300]

total = 0
for amount in sales:
    total += amount
    if amount > 3000:
        print(f"  High-value sale: ₱{amount:,}")

print(f"Total: ₱{total:,}")
print(f"Average: ₱{total/len(sales):,.2f}")

# List comprehension — the "pythonic" one-liner loop
vat_inclusive = [round(s * 1.12, 2) for s in sales]
print("With 12% VAT:", vat_inclusive)

In [ ]:
# Functions + error handling — the building blocks of reusable data code
def compute_discount(amount, customer_type="regular"):
    """Return the discounted amount based on customer type."""
    rates = {"regular": 0.0, "senior": 0.20, "pwd": 0.20, "member": 0.05}
    if customer_type not in rates:
        raise ValueError(f"Unknown customer type: {customer_type}")
    return round(amount * (1 - rates[customer_type]), 2)

print(compute_discount(1000))             # 1000.0
print(compute_discount(1000, "senior"))   # 800.0

# try/except keeps pipelines and analyses from crashing on bad input
try:
    compute_discount(1000, "vip")
except ValueError as e:
    print("Handled gracefully →", e)

## Loading Data from Every Source: CSV, TXT, Parquet & API (≈25 min)
Real projects pull data from many formats. Pandas gives you one interface: everything ends up as a **DataFrame**.

| Format | Best for | Reader |
|---|---|---|
| **CSV** | Universal exchange, spreadsheets | `pd.read_csv()` |
| **TXT** | Logs, raw text | `open()` + parsing (often regex) |
| **Parquet** | Big data, fast columnar analytics | `pd.read_parquet()` |
| **API (JSON)** | Live data from web services | `requests` + `pd.json_normalize()` |

In [ ]:
import pandas as pd

# 1) CSV
sales = pd.read_csv("data/sales_data.csv")
print("CSV     →", sales.shape)

# 2) Parquet — note: column types are preserved (timestamps stay timestamps!)
tx = pd.read_parquet("data/transactions.parquet")
print("Parquet →", tx.shape, "| timestamp dtype:", tx["timestamp"].dtype)

# 3) TXT
with open("data/server_logs.txt") as f:
    log_lines = f.readlines()
print("TXT     →", len(log_lines), "log lines")

sales.head(3)

In [ ]:
# 4) API — the standard pattern with the `requests` library
import requests, json
import pandas as pd

# Real public API (needs internet). JSONPlaceholder is a free practice API.
try:
    resp = requests.get("https://jsonplaceholder.typicode.com/users", timeout=10)
    resp.raise_for_status()                      # error if status != 200
    users = pd.json_normalize(resp.json())      # nested JSON → flat table
    print("Live API call OK — status:", resp.status_code)
except Exception as e:
    print("No internet? Using the saved mock API response instead.", e)
    with open("data/customers_api.json") as f:
        users = pd.json_normalize(json.load(f)["data"])

users.head(3)

In [ ]:
# Our saved mock API response — same pattern, local file
import json
import pandas as pd

with open("data/customers_api.json") as f:
    payload = json.load(f)

print("status:", payload["status"], "| count:", payload["count"])
customers = pd.json_normalize(payload["data"])
customers.head()

## Data Cleaning (≈25 min)
Analysts and engineers spend **50–80% of their time cleaning data**. The classic checklist:
1. **Missing values** → fill, drop, or flag
2. **Duplicates** → remove
3. **Inconsistent categories** → standardize (`" ncr "` vs `"NCR"`)
4. **Wrong data types** → convert (dates stored as text, numbers stored as strings)
5. **Invalid values** → detect with rules or regex

In [ ]:
import pandas as pd
sales = pd.read_csv("data/sales_data.csv")

# --- Diagnose first ---
print("Shape:", sales.shape)
print("\nMissing values per column:")
print(sales.isna().sum())
print("\nDuplicate rows:", sales.duplicated().sum())
print("\nRegion values (note the messy ' ncr '):", sales["region"].unique())

In [ ]:
# --- Clean step by step ---
clean = sales.copy()

# 1) Duplicates
clean = clean.drop_duplicates()

# 2) Standardize categories: strip spaces, uppercase
clean["region"] = clean["region"].str.strip().str.upper()

# 3) Missing unit_price → recover it from total_amount / quantity
mask = clean["unit_price"].isna()
clean.loc[mask, "unit_price"] = (clean.loc[mask, "total_amount"] / clean.loc[mask, "quantity"]).round(2)

# 4) Fix mixed date formats (2025-03-01 vs 2025/03/01) → real datetime
clean["order_date"] = pd.to_datetime(clean["order_date"], format="mixed")

# 5) Flag invalid emails with regex
clean["email_valid"] = clean["customer_email"].str.match(r"^[\w\.\-]+@[\w\-]+\.[\w\.]+$")

print("After cleaning:", clean.shape)
print("Missing:", clean.isna().sum().sum(), "| Duplicates:", clean.duplicated().sum())
print("Regions:", clean["region"].unique())
print("Invalid emails flagged:", (~clean["email_valid"]).sum())
clean.to_csv("data/sales_clean.csv", index=False)   # save for later sections
clean.head(3)

## Regular Expressions (Regex) — Pattern Matching for Text (≈25 min)
Regex lets you **find, validate, and extract** patterns in text — emails, phone numbers, log entries, IDs. The `re` module is built into Python, and pandas has `.str.extract()` / `.str.contains()` built on top of it.

**Cheat sheet:**
| Pattern | Matches |
|---|---|
| `\d` | a digit (0–9) |
| `\w` | a word character (letter, digit, `_`) |
| `\s` | whitespace |
| `+` / `*` | one-or-more / zero-or-more |
| `{n,m}` | between *n* and *m* repeats |
| `[abc]` | any one of a, b, c |
| `^` / `$` | start / end of string |
| `( )` | capture group (extract this part) |

In [ ]:
import re

# --- Extract structured fields from raw log lines ---
with open("data/server_logs.txt") as f:
    logs = f.readlines()

print("Raw log line:")
print(logs[0])

# One pattern, four capture groups: timestamp, level, status, response time
pattern = r"^([\d\-]+ [\d:]+) \[(\w+)\] client=([\d\.]+) endpoint=(\S+) status=(\d{3}) response_time=(\d+)ms"

m = re.match(pattern, logs[0])
print("\nParsed:", m.groups())

# Parse ALL lines into a DataFrame — regex turns unstructured text into a table!
import pandas as pd
parsed = [re.match(pattern, line).groups() for line in logs if re.match(pattern, line)]
log_df = pd.DataFrame(parsed, columns=["timestamp", "level", "client_ip", "endpoint", "status", "response_ms"])
log_df["response_ms"] = log_df["response_ms"].astype(int)
log_df["status"] = log_df["status"].astype(int)
log_df.head()

In [ ]:
# --- Validating & extracting with regex ---
import re

emails = ["nina@dep.org.ph", "invalid-email", "juan.cruz@gmail.com", "test@", "ana_reyes@yahoo.com"]
email_pattern = r"^[\w\.\-]+@[\w\-]+\.[\w\.]+$"

for e in emails:
    ok = "✅ valid" if re.match(email_pattern, e) else "❌ invalid"
    print(f"{e:30s} {ok}")

# Extract PH mobile numbers from free text
text = "Contact us at 09171234567 or 09989876543. Landline: 8123-4567."
print("\nMobile numbers found:", re.findall(r"09\d{9}", text))

# re.sub — find & replace (mask sensitive digits)
masked = re.sub(r"(09\d{2})\d{5}(\d{2})", r"\1*****\2", text)
print("Masked:", masked)

In [ ]:
# --- Regex inside pandas: clean a real column ---
import pandas as pd
sales = pd.read_csv("data/sales_data.csv")

# Flag invalid emails using .str.match()
valid = sales["customer_email"].str.match(r"^[\w\.\-]+@[\w\-]+\.[\w\.]+$")
print(f"Invalid emails found: {(~valid).sum()} out of {len(sales)} rows")
sales.loc[~valid, "customer_email"].head()

## Part 5 — Transforming & Aggregating Data (≈25 min)
The analyst's power tools: **groupby**, **pivot tables**, and **merges** — the pandas versions of Excel's SUMIFS, PivotTable, and VLOOKUP.

In [ ]:
import pandas as pd
sales = pd.read_csv("data/sales_clean.csv", parse_dates=["order_date"])

# GROUPBY — like a pivot table's row summary
summary = (sales
    .groupby("region")
    .agg(orders=("order_id", "count"),
         total_revenue=("total_amount", "sum"),
         avg_order=("total_amount", "mean"))
    .round(2)
    .sort_values("total_revenue", ascending=False))
summary

In [ ]:
# PIVOT TABLE — two dimensions at once: region × product
pivot = pd.pivot_table(sales, values="total_amount",
                       index="region", columns="product",
                       aggfunc="sum", fill_value=0).round(0)
pivot

In [ ]:
# MERGE — combine sales with the customer API data (like VLOOKUP / SQL JOIN)
import json
with open("data/customers_api.json") as f:
    customers = pd.json_normalize(json.load(f)["data"])

# Simulate a shared key: map each order to a customer_id
sales["customer_id"] = (sales.reset_index().index % 60) + 1
enriched = sales.merge(customers[["customer_id", "city", "loyalty_points"]],
                       on="customer_id", how="left")
print("Before merge:", sales.shape, "→ After merge:", enriched.shape)
enriched[["order_id", "region", "total_amount", "city", "loyalty_points"]].head()

## Part 6 — Descriptive Statistics (≈15 min)
Summarize before you conclude: **mean vs. median** (skew!), **spread** (std), and **outliers**.

In [ ]:
import pandas as pd
sales = pd.read_csv("data/sales_clean.csv")

print(sales["total_amount"].describe().round(2))
print()
mean, median = sales["total_amount"].mean(), sales["total_amount"].median()
print(f"Mean ₱{mean:,.2f} vs Median ₱{median:,.2f} → mean > median means RIGHT-SKEWED (big orders pull the average up)")

# Outliers with the IQR rule
q1, q3 = sales["total_amount"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
outliers = sales[sales["total_amount"] > upper]
print(f"\nIQR outlier threshold: ₱{upper:,.2f} → {len(outliers)} unusually large orders")

## Basic Charts with Matplotlib & Seaborn (≈30 min)
**Choosing the right chart:**
| Question | Chart |
|---|---|
| Compare categories | **Bar** |
| Trend over time | **Line** |
| Distribution of one variable | **Histogram** |
| Relationship of two variables | **Scatter** |
| Correlations across many variables | **Heatmap** |
| Share of a whole (few categories) | **Pie** |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sales = pd.read_csv("data/sales_clean.csv", parse_dates=["order_date"])
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1) BAR — revenue by region
by_region = sales.groupby("region")["total_amount"].sum().sort_values(ascending=False)
axes[0,0].bar(by_region.index, by_region.values, color="steelblue")
axes[0,0].set_title("Total Revenue by Region")
axes[0,0].set_ylabel("Revenue (₱)")

# 2) LINE — monthly revenue trend
monthly = sales.set_index("order_date").resample("ME")["total_amount"].sum()
axes[0,1].plot(monthly.index, monthly.values, marker="o", color="darkgreen")
axes[0,1].set_title("Monthly Revenue Trend (2025)")
axes[0,1].tick_params(axis="x", rotation=45)

# 3) HISTOGRAM — distribution of order values
axes[1,0].hist(sales["total_amount"], bins=30, color="coral", edgecolor="white")
axes[1,0].set_title("Distribution of Order Amounts")
axes[1,0].set_xlabel("Order total (₱)")

# 4) SCATTER — quantity vs total
axes[1,1].scatter(sales["quantity"], sales["total_amount"], alpha=0.4, color="purple")
axes[1,1].set_title("Quantity vs Order Total")
axes[1,1].set_xlabel("Quantity"); axes[1,1].set_ylabel("Total (₱)")

plt.tight_layout()
plt.show()

In [ ]:
# Seaborn shines for statistical plots — heatmap + boxplot
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sales = pd.read_csv("data/sales_clean.csv", parse_dates=["order_date"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Correlation heatmap (numeric columns only)
corr = sales[["quantity", "unit_price", "total_amount"]].corr()
sns.heatmap(corr, annot=True, cmap="Blues", fmt=".2f", ax=axes[0])
axes[0].set_title("Correlation Heatmap")

# Boxplot — order totals per region (spot outliers instantly)
sns.boxplot(data=sales, x="region", y="total_amount", ax=axes[1], hue="region", legend=False, palette="Set2")
axes[1].set_title("Order Totals by Region")

plt.tight_layout()
plt.show()

## Part 8 — Dates & Time Series (≈15 min)

In [ ]:
import pandas as pd
sales = pd.read_csv("data/sales_clean.csv", parse_dates=["order_date"])

# Extract date parts
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()

# Resample = group by time period
ts = sales.set_index("order_date").resample("ME")["total_amount"].sum()
print("Monthly revenue:")
print(ts.round(0).head())

# Which day of the week sells the most?
weekday_sales = sales.groupby("weekday")["total_amount"].sum().sort_values(ascending=False)
weekday_sales.round(0)

---
# 🧪 HANDS-ON LAB (4 hours)
Work through the exercises **in order**. Each block has a difficulty tag and an estimated time. Write your code in the empty cells. Solutions discussion happens at the end of the session.

### Block A — Warm-up (⭐ 45 min)
1. Load `data/sales_clean.csv`. How many orders came from **Cebu**?
2. What is the **most sold product** by total quantity?
3. Count how many customer emails are from **yahoo.com** (hint: `.str.contains`).
4. What was the **single largest order** and what product was it?

In [ ]:
# Block A — your code here


### Block B — Cleaning + Regex (⭐⭐ 60 min)
1. Reload the **raw** `data/sales_data.csv` and redo the full cleaning pipeline yourself (dedupe → standardize region → fix prices → parse dates).
2. Using regex, extract the **email domain** (e.g., `gmail.com`) into a new column, then count orders per domain.
3. Using regex on `phone`, keep only rows with a valid PH mobile format `09XXXXXXXXX` (exactly 11 digits).
4. Parse `data/server_logs.txt` into a DataFrame and report: how many **ERROR** lines are there, and which **endpoint** has the highest average response time?

In [ ]:
# Block B — your code here


### Block C — Aggregation + Visualization (⭐⭐ 60 min)
1. Build a pivot table of **average order value** per region × month.
2. Create a **bar chart** of revenue per product and a **line chart** of weekly revenue (`resample('W')`).
3. Merge in the customer API data and chart average `loyalty_points` per city.
4. Make one chart of your own choice that reveals something interesting — and write one sentence explaining the insight.

In [ ]:
# Block C — your code here


### Block D — Mini-Capstone: Analysis Report (⭐⭐⭐ 75 min)
**Scenario:** The store manager asks: *"Which region should we prioritize next year and why?"*

Deliver, inside this notebook:
1. A short **executive summary** (markdown cell, 3–5 sentences).
2. At least **3 charts** supporting your recommendation.
3. One table of **key numbers** (revenue, average order, order count per region).
4. One caveat about **data quality** you found along the way.

*Rubric: correctness of numbers (40%), chart clarity (30%), narrative quality (30%).*

In [ ]:
# Block D — your capstone here


---
---
### © Data Engineering Pilipinas (DEP). All rights reserved.
*This course material was created by and is the property of **Data Engineering Pilipinas**. It may not be copied, distributed, modified, or used — in whole or in part — without prior written consent from Data Engineering Pilipinas.*